# Face swap — single-pass Krea generation, NO masking

**No masks, no crop, no stitch, no compositing.** The whole photo goes to Krea2
once, with one instruction, and the model's own render ships exactly as
generated.

This is a different route from the earlier face-swap notebook. That one used
`crop_stitch`, which regenerates a small face crop and pastes it back through a
soft mask -- and that stitch boundary is what produced the ghosted/haloed
headwear. `full_frame` is no better: it builds its own freeze mask and does
LAB-match + feather compositing.

This notebook uses `run_simple_full_body()` with `raw_model=True`, the only
route in this codebase with genuinely zero post-generation compositing:

| stage | status |
|---|---|
| crop / stitch / paste | never runs (different route) |
| `face_refine` (feathered_soft_composite) | **off** via `simple_full_body_face_refine=False` |
| `body_restore` | off via `raw_model` |
| LAB skin wash | off via `raw_model` |
| `skin_repaint` | off via `raw_model` |
| head-scale clamp / procrustes warp | never runs (different route) |

The tradeoff, stated honestly: with no mask, clothing/hair/background are
preserved by the *prompt* and by img2img seeding from the source latent, not by
a structural guarantee. That is the point -- it is what "just Krea generation"
means -- but it is why the prompt below is explicit about what must not change.

**3 cells: Setup → Upload → Run.**

## 1 · Setup

In [ ]:
from pathlib import Path
import subprocess, os, sys

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
BRANCH = "face-swap-no-mask"

# GPU check WITHOUT importing torch.
#
# `import torch` pulls numpy into THIS kernel. The setup below installs
# simple-lama-inpainting, which downgrades numpy, and then force-reinstalls
# numpy back to 2.3.4 on disk. If numpy was already imported here first, the
# kernel keeps the OLD compiled extension in memory while the on-disk .py
# files are NEW -- and the first fresh submodule import during a render then
# fails with:
#     ImportError: cannot import name '_slice' from 'numpy._core.umath'
# Keeping this kernel free of numpy until after setup means no runtime
# restart is needed. GPU-confirmed failure mode, not hypothetical.
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit(
        "No GPU detected. Runtime -> Change runtime type -> GPU, then Run all."
    )
print(f"GPU: {gpu.stdout.strip().splitlines()[0]}")

from google.colab import drive
drive.mount("/content/drive")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"],
    check=True,
)

os.chdir(REPO)
subprocess.run(["bash", "scripts/setup_colab.sh", "--krea2"], check=True,
               cwd=str(REPO))

# Verify numpy is healthy in a FRESH interpreter -- this kernel deliberately
# has not imported it, so a clean subprocess is the honest check.
check = subprocess.run(
    [sys.executable, "-c",
     "import numpy, numpy._core.strings; print(numpy.__version__)"],
    capture_output=True, text=True,
)
if check.returncode != 0:
    print(check.stderr)
    raise SystemExit(
        "numpy is in a broken state after setup (see error above). Re-run "
        "this cell; if it persists, Runtime -> Restart session and run again."
    )
print(f"numpy OK: {check.stdout.strip()}")
print("Setup complete.")


## 2 · Upload your pairs

Two file pickers, each **multi-select** -- pick all your body photos at once,
then all your face photos at once.

Order matters: the Nth body is paired with the Nth face, sorted by filename. The
cell prints the resulting pairing table so you can check it before running
anything. Naming them `01.png, 02.png, ...` on both sides makes this exact.

Files are written to disk immediately and cell 3 reads from disk, so the two
cells share no in-memory state -- restarts and re-runs cannot break it.

> Uses Colab's native uploader rather than `ipywidgets.FileUpload` buttons:
> those render in Colab but their contents often never sync back to the
> kernel, which is why an earlier version silently saved nothing.

In [ ]:
import io
import shutil
from pathlib import Path

from google.colab import files
from PIL import Image

UPLOAD_DIR = Path("/content/face_swap_uploads")
CLEAR_PREVIOUS = True   # wipe earlier uploads so stale pairs cannot leak in

if CLEAR_PREVIOUS and UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("STEP 1 of 2 — select ALL your BODY / scene photos (multi-select)")
body_uploads = files.upload()

print()
print("STEP 2 of 2 — select ALL your FACE / identity photos (multi-select)")
face_uploads = files.upload()

body_names = sorted(body_uploads)
face_names = sorted(face_uploads)

if len(body_names) != len(face_names):
    raise RuntimeError(
        f"Got {len(body_names)} body photo(s) and {len(face_names)} face "
        "photo(s) -- these must match 1:1. Re-run this cell with matching "
        "sets."
    )
if not body_names:
    raise RuntimeError("Nothing was uploaded -- re-run this cell.")

print()
print("Pairing (check this is what you intended):")
for i, (bn, fn) in enumerate(zip(body_names, face_names), start=1):
    Image.open(io.BytesIO(body_uploads[bn])).convert("RGB").save(
        UPLOAD_DIR / f"body_{i:02d}.png"
    )
    Image.open(io.BytesIO(face_uploads[fn])).convert("RGB").save(
        UPLOAD_DIR / f"face_{i:02d}.png"
    )
    print(f"  {i:2d}.  body={bn!r}   <->   face={fn!r}")

print(f"\n{len(body_names)} pair(s) saved to {UPLOAD_DIR}. Now run cell 3.")


## 3 · Run all pairs

Loads the model once, then runs each pair one at a time, showing each result as
it finishes.

Watch the log for these two lines -- they confirm the mask-free path actually
ran:

```
[krea2 body_route] resolved_mode=simple_full_body ...
[krea2 raw_model] body_restore + LAB wash + skin_repaint all DISABLED ...
```

You should **not** see `crop_stitch`, `procrustes`, `feathered_soft_composite`,
`face_refine`, or `skin_harm` anywhere.

In [ ]:
from pathlib import Path
import importlib.util
import sys, os, time

# Fail fast and legibly if numpy is in the mixed old-extension/new-files
# state that a mid-session reinstall leaves behind (see cell 1). Without
# this the error surfaces 30+ seconds into a render, from deep inside an
# unrelated import chain, and reads as a pipeline bug rather than an
# environment one.
try:
    import numpy, numpy._core.strings  # noqa: F401
except ImportError as _np_exc:
    raise SystemExit(
        f"numpy is in a mixed state in this kernel ({_np_exc}). "
        "Runtime -> Restart session, then run cells 1, 2 and 3 again."
    ) from _np_exc

REPO = Path("/content/headswap_V2")
UPLOAD_DIR = Path("/content/face_swap_uploads")

# Reads from DISK, not from cell 2's in-memory widgets. An earlier version
# depended on the widget objects still existing in the kernel, which broke
# whenever the runtime restarted or a stale copy of the cell was run.
pairs, half_filled = [], []
for i in range(1, 14):
    bp = UPLOAD_DIR / f"body_{i:02d}.png"
    fp = UPLOAD_DIR / f"face_{i:02d}.png"
    if not bp.exists() and not fp.exists():
        continue
    if not bp.exists() or not fp.exists():
        half_filled.append(i)
        continue
    pairs.append((i, bp, fp))

if half_filled:
    print(f"Skipping pairs with only one side uploaded: {half_filled}")
if not pairs:
    raise RuntimeError(
        f"No complete pairs found in {UPLOAD_DIR}. Run cell 2 and upload at "
        "least one Body + Face pair (a ✓ appears next to each button once "
        "the file is on disk)."
    )

spec = importlib.util.spec_from_file_location(
    "colab_env", REPO / "scripts" / "colab_env.py"
)
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
colab_env.apply_env(colab_env.default_paths(use_drive=True))
colab_env.ensure_import_path(REPO)

from headswap.config import load_config
from headswap.pipelines.krea2 import Krea2IdentityEditPipeline
from PIL import Image
from IPython.display import display, Markdown

# Full-frame, single-pass prompt. The model sees the WHOLE photo and
# regenerates it in one go -- there is no mask holding anything still, so
# everything that must stay the same has to be said here.
# Mirrors T4's proven structure: REPLACEMENT FIRST, in imperative form,
# then one short preservation clause. Length and ordering are not
# cosmetic here -- CHECKPOINT-10 measured that a clause buried after
# prohibitions is simply not acted on, and CHECKPOINT-11 measured that
# prompt LENGTH alone moves how much of the face the model rebuilds.
#
# The previous version of this prompt was 1574 chars, almost entirely
# preservation instructions, with "take only facial identity from the
# second image" sitting dead last. Result: the model preserved
# everything -- including the original face. Identity did not transfer
# at all. That is the documented failure mode, not a model limitation.
#
# "with none of the first person's face remaining" is the forcing phrase
# T4 relies on ("...none of the first person's head remaining"); without
# something that strong, img2img at denoise=0.85 just keeps what is
# already in the source latent.
FACE_SWAP_PROMPT = (
    "One change: replace the face from the first image with the face "
    "from the second image completely -- the bone structure, jawline, "
    "brows, eyes, nose, mouth and facial skin, exactly as they appear "
    "in the second image, with none of the first person's face "
    "remaining, and at the same apparent age as the second person. "
    "Keep the first person's own hair, headwear, clothing, pose, body, "
    "background and lighting exactly as they are."
)

base_cfg = load_config(str(REPO / "configs" / "krea2_identity_edit.yaml"))
cfg = dict(base_cfg)
cfg.update({
    # --- route selection: force the single-pass, mask-free path -----------
    # simple_full_body is picked for a single-face photo when the body route
    # is enabled. Lowering the "enough body visible" floor from 0.38 means a
    # bust/portrait crop takes this route too, instead of falling back to
    # crop_stitch (which masks).
    "enable_body_route": True,
    "simple_path_below_face_frac_min": 0.05,
    "enable_lighting_route": False,

    # --- identity: same architecture as head-swap production -------------
    # This route is head-swap's T4 pipeline with a different prompt. Its
    # sampling values (ref_boost=5.5, denoise=0.85, cfg=1.8, seed=46) and
    # its output resolution are deliberately NOT overridden -- they are a
    # tuned recipe, and a face swap is the same job with different words.
    #
    # face_refine is left ON, exactly as head-swap production leaves it.
    # chain.py's skip_refine=True does NOT disable it: it sets
    # refine_max_face_frac=0.25, i.e. "refine when the face is under 25% of
    # frame, skip when it is already high-resolution". On a full-body shot
    # the face is ~8% of frame -- about 84px at 1024 output -- and identity
    # cannot survive in 84px from the main pass alone. face_refine
    # re-renders the head region at full resolution, which is what actually
    # carries identity here.
    #
    # An earlier version of this notebook set
    # simple_full_body_face_refine=False, reading "no masks" as maximally as
    # possible. It does composite (feathered_soft_composite over the head
    # box), but it is head-local, it is what head-swap production has always
    # run, and turning it off destroyed identity -- the model had 84px to
    # work with. Mask-freedom was traded for a broken result.
    "simple_full_body_refine_max_face_frac": 0.25,
    # Measures the ORIGINAL's expression and states it as a fact in the
    # prompt. Log showed "[krea2 expression] no hint - disabled" while
    # expression drifted off the original. Prompt text, not a mask.
    "expression_prompt_hint": True,

    # --- the prompt (overrides the built-in head-swap text entirely) ------
    "simple_full_body_prompt": FACE_SWAP_PROMPT,

    # --- no whole-frame compositing --------------------------------------
    # raw_model ships the model's own render: no body_restore, no LAB skin
    # wash, no skin_repaint. The only composite left is the head-local
    # face_refine, same as head-swap production.
    "simple_full_body_raw_model": True,
    # face_refine stays ON -- see the identity note above.

    # --- head-swap-specific behaviour that must NOT apply to a face swap --
    # This route normally REPLACES headwear with the donor's hair. A face
    # swap keeps it.
    "simple_full_body_remove_headwear": False,
    # Experimental garment work from the head-swap branch -- all off.
    "simple_full_body_protect_garments": False,
    "skip_skin_clause_when_covered": False,
    "simple_full_body_garment_containment": False,
    "simple_full_body_restore_stripped_garment": False,
})

CACHE_DIR = Path("/content/.cache/headswap_v2")
OUT_DIR = Path("/content/face_swap_results")
for d in (CACHE_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"{len(pairs)} complete pair(s) found on disk. Loading model once, "
      "then running them one at a time.\n")

from headswap.preprocess import pil_to_rgb_np, select_face_box


def _has_face(path):
    """A legs-only or object photo silently produced a no-op pair before."""
    try:
        box, _ = select_face_box(
            pil_to_rgb_np(Image.open(path).convert("RGB")),
            CACHE_DIR, index=0, policy="largest",
        )
        return box is not None
    except Exception:
        return True   # never block a render on the detector failing


pipe = Krea2IdentityEditPipeline(cfg=cfg, cache_dir=CACHE_DIR)

t0 = time.perf_counter()
for n, (idx, body_path, face_path) in enumerate(pairs, start=1):
    body = Image.open(body_path).convert("RGB")
    face = Image.open(face_path).convert("RGB")
    pair_out = OUT_DIR / f"pair_{idx:02d}"
    t = time.perf_counter()
    if not _has_face(body_path):
        print(f"Pair {idx} SKIPPED: no face detected in the BODY photo "
              f"({body_path.name}) -- nothing to swap.")
        continue
    if not _has_face(face_path):
        print(f"Pair {idx} SKIPPED: no face detected in the FACE photo "
              f"({face_path.name}) -- no identity to take.")
        continue
    try:
        res = pipe.run(body, face, out_dir=pair_out)
    except Exception as exc:                     # noqa: BLE001
        print(f"Pair {idx} FAILED: {type(exc).__name__}: {exc}")
        continue
    route = (res.meta or {}).get("edit_mode", "?")
    display(Markdown(
        f"### Pair {idx} &nbsp;·&nbsp; {time.perf_counter() - t:.0f}s "
        f"&nbsp;·&nbsp; route=`{route}` &nbsp;·&nbsp; [{n}/{len(pairs)}]"
    ))
    if route != "simple_full_body":
        print(f"  WARNING: pair {idx} did NOT take the mask-free route "
              f"(route={route}). Its result went through compositing.")
    display(res.image)

print(f"\nAll {len(pairs)} pair(s) done in {time.perf_counter() - t0:.0f}s total.")
print(f"Saved under {OUT_DIR}")
